# From examples to a living spec

Run one section at a time; stop to vote in the embedded view.

Needs `suggest-search`, Node, and an authenticated `codex` CLI in the kernel's environment.

In [ ]:
from pathlib import Path
import sys

# Use this checkout from demos/, spytial-py/, or spytial-org/.
cwd = Path.cwd()
repo = next((p for p in (cwd, cwd.parent, cwd / "spytial-py")
             if (p / "spytial" / "suggest" / "_session.py").is_file()), None)
if repo is None:
    raise RuntimeError("Open this notebook from the spytial-py checkout or its demos folder.")
sys.path.insert(0, str(repo))

from dataclasses import dataclass
from typing import Optional
from hypothesis import strategies as st
from IPython.display import display
import spytial
from spytial.suggest import suggest, SuggestSession, SpecUnavailableError

In [ ]:
@dataclass
class Tree:
    value: int
    left: Optional["Tree"] = None
    right: Optional["Tree"] = None

examples = [
    Tree(8, Tree(3), Tree(10)),
    Tree(8, Tree(3, Tree(1), Tree(6)), Tree(10, None, Tree(14))),
]

# Generate small trees without cycles or shared children.
def trees(depth):
    children = st.none() if depth == 0 else st.one_of(st.none(), trees(depth - 1))
    return st.builds(Tree, value=st.integers(0, 20), left=children, right=children)

strategy = trees(2)
requested_rules = ["Put children below their parents", "Align children"]  # Or [].

## Explore

This creates a session. Rerun only `display(session)` to return to the same one.

In [ ]:
session = suggest(
    Tree,
    rules=requested_rules,
    examples=examples,
    strategy=strategy,
    max_examples=10,
)
spec = session.spec

In [ ]:
display(session)

Drag, then **upvote**, **downvote**, or **unsure**. **History** lets you revisit votes.

## I'm done

Click the button above, or run this. It pauses and checkpoints; it does not apply the spec.

In [ ]:
checkpoint = session.done()
{
    "paused": session.paused,
    "result": checkpoint["status"],
    "remaining candidates": len(session.survivors),
    "applied": spec.applied,
}

## Apply and draw

If the result needs work, use **Resume** and **History**, then retry this cell.

In [ ]:
try:
    spec = session.apply()
except SpecUnavailableError as error:
    print(error)
else:
    spytial.diagram(examples[1])

## Keep refining

The same session keeps your votes. Edit the request below.

In [ ]:
session.resume()
display(session)

In [ ]:
session.revise("Keep left children left of their parent, and right children right of it")

Vote again above, then draw the same tree. This uses the same `spec` object; earlier drawings stay unchanged.

In [ ]:
try:
    spytial.diagram(examples[1], spec=spec)
except SpecUnavailableError as error:
    print(error)

## Save and close the controls

Run this only when leaving. `close()` stops the embedded controls; it does not unapply the spec.

In [ ]:
save_path = Path("tree-session.json")
session.save(save_path)
session.close()
print(f"Controls closed. Saved to {save_path.resolve()}")

A supported spec still draws after closing:

In [ ]:
try:
    spytial.diagram(examples[1], spec=spec)
except SpecUnavailableError as error:
    print(error)

## Reopen the saved session

Reload to continue voting. Then rerun **Apply and draw** to bind the restored session to `Tree`.

In [ ]:
session = SuggestSession.load(Tree, save_path, strategy=strategy, max_examples=10)
spec = session.spec
session.resume()
display(session)